# COVID-19 Global & India: Exploratory Data Analysis (EDA)

**Portfolio project** using Python, Pandas, NumPy, Matplotlib, and Seaborn.

Questions explored: How did reported cases change over time? Which countries have the highest reported totals in the selected snapshot? How do cases and deaths relate? What patterns appear in India's daily reported cases?

> This is a historical/public-health data exploration project, not medical advice. Reported cases depend on testing and reporting practices; comparisons should be interpreted carefully.

## 1. Import libraries and load the dataset
The notebook downloads the compact Our World in Data COVID-19 dataset directly, so you do not need to manually upload a large CSV. The source and data notes are in `README.md`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

DATA_URL = "https://catalog.ourworldindata.org/garden/covid/latest/compact/compact.csv"
df = pd.read_csv(DATA_URL, low_memory=False)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

## 2. Clean and inspect the data
Check column names, data types, missing values, and duplicate rows. The dataset schema may evolve, so this cell checks which useful columns are available.

In [ ]:
print("Data types and non-null counts:")
df.info()

print("\nMissing values (top 15):")
display(df.isna().sum().sort_values(ascending=False).head(15))

print("Duplicate rows:", df.duplicated().sum())

if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

print("\nPreview after basic cleaning:")
display(df.head())

## 3. Summary statistics
Use Pandas and NumPy to inspect the numerical data. Values depend on the source dataset snapshot.

In [ ]:
display(df.describe(include="all").T.head(20))

for col in ["total_cases", "new_cases", "total_deaths", "new_deaths", "people_vaccinated"]:
    if col in df.columns:
        values = pd.to_numeric(df[col], errors="coerce").dropna()
        if not values.empty:
            print(f"{col}: mean={values.mean():,.2f}, median={values.median():,.2f}, max={values.max():,.2f}")

## 4. India: daily reported cases over time
A seven-day rolling average smooths day-to-day reporting variation. This chart is shown only if the expected columns exist.

In [ ]:
required = {"location", "date", "new_cases"}
if required.issubset(df.columns):
    india = df.loc[df["location"].astype(str).str.casefold().eq("india")].copy()
    india["new_cases"] = pd.to_numeric(india["new_cases"], errors="coerce")
    india = india.dropna(subset=["date"]).sort_values("date")
    india["new_cases_7day_avg"] = india["new_cases"].rolling(7, min_periods=1).mean()
    if not india.empty:
        fig, ax = plt.subplots()
        ax.plot(india["date"], india["new_cases"], alpha=0.35, label="Daily reported cases")
        ax.plot(india["date"], india["new_cases_7day_avg"], linewidth=2, label="7-day rolling average")
        ax.set(title="India: Daily Reported COVID-19 Cases", xlabel="Date", ylabel="Reported cases")
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("No India rows found in the current dataset.")
else:
    print("Required columns not available:", required - set(df.columns))

## 5. Country comparison using the latest available date per country
This avoids comparing different dates for countries when the dataset contains time series.

In [ ]:
if {"location", "date", "total_cases"}.issubset(df.columns):
    country_data = df.copy()
    country_data["total_cases"] = pd.to_numeric(country_data["total_cases"], errors="coerce")
    country_data = country_data.dropna(subset=["date", "total_cases"])
    latest = country_data.sort_values("date").groupby("location", as_index=False).tail(1)
    latest = latest[~latest["location"].astype(str).str.contains("income|world|union|international", case=False, na=False)]
    top10 = latest.nlargest(10, "total_cases").sort_values("total_cases")
    display(top10[["location", "date", "total_cases"]])
    plt.figure(figsize=(10, 6))
    sns.barplot(data=top10, x="total_cases", y="location")
    plt.title("Top 10 Locations by Reported Total COVID-19 Cases (latest available row)")
    plt.xlabel("Reported total cases")
    plt.ylabel("Location")
    plt.tight_layout()
    plt.show()
else:
    print("Required columns not available for country comparison.")

## 6. Cases and deaths relationship
A scatter plot can reveal an association between reported totals. It does **not** establish causation or a person's risk.

In [ ]:
if {"total_cases", "total_deaths"}.issubset(df.columns):
    relation = df[[c for c in ["location", "date", "total_cases", "total_deaths"] if c in df.columns]].copy()
    relation["total_cases"] = pd.to_numeric(relation["total_cases"], errors="coerce")
    relation["total_deaths"] = pd.to_numeric(relation["total_deaths"], errors="coerce")
    relation = relation.dropna(subset=["total_cases", "total_deaths"])
    if "date" in relation.columns:
        relation = relation.sort_values("date").groupby("location", as_index=False).tail(1)
    relation = relation[(relation["total_cases"] > 0) & (relation["total_deaths"] >= 0)]
    if len(relation) > 1:
        plt.figure(figsize=(9, 6))
        sns.scatterplot(data=relation, x="total_cases", y="total_deaths", alpha=0.65)
        plt.xscale("log")
        plt.yscale("symlog")
        plt.title("Reported Total Cases vs Reported Total Deaths by Location")
        plt.tight_layout()
        plt.show()
        print("Correlation (latest available location totals):", relation[["total_cases", "total_deaths"]].corr().iloc[0, 1])
    else:
        print("Not enough rows for the relationship plot.")
else:
    print("Case/death columns are not available in this snapshot.")

## 7. Key findings — write after running
Record 3–5 findings from your actual charts, such as the date range covered, visible peaks in India's reported cases, countries with the largest reported totals in this snapshot, and any missing-data limitations.

Do not invent results. Reported counts can be affected by testing, reporting delays, revisions, and differences between countries.